In [5]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
import math
import time

In [6]:
import joblib

model = joblib.load("confidence_xgboost_model.pkl")

print("Model loaded successfully")
print("Model type:", type(model))
print("Number of features:", model.n_features_in_)
print("Classes:", model.classes_)

Model loaded successfully
Model type: <class 'xgboost.sklearn.XGBClassifier'>
Number of features: 18
Classes: [0 1 2]


In [7]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [8]:
def extract_features(landmarks):

    # ==========================================
    # MediaPipe landmark indexes
    # ==========================================

    NOSE = 0

    LEFT_EYE = 2
    RIGHT_EYE = 5

    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12

    LEFT_WRIST = 15
    RIGHT_WRIST = 16

    LEFT_HIP = 23
    RIGHT_HIP = 24


    # ==========================================
    # Get landmarks
    # ==========================================

    nose = landmarks[NOSE]

    left_eye = landmarks[LEFT_EYE]
    right_eye = landmarks[RIGHT_EYE]

    left_shoulder = landmarks[LEFT_SHOULDER]
    right_shoulder = landmarks[RIGHT_SHOULDER]

    left_wrist = landmarks[LEFT_WRIST]
    right_wrist = landmarks[RIGHT_WRIST]

    left_hip = landmarks[LEFT_HIP]
    right_hip = landmarks[RIGHT_HIP]


    # ==========================================
    # Calculate centers
    # ==========================================

    eye_center_x = (
        left_eye.x + right_eye.x
    ) / 2

    eye_center_y = (
        left_eye.y + right_eye.y
    ) / 2


    shoulder_center_x = (
        left_shoulder.x + right_shoulder.x
    ) / 2

    shoulder_center_y = (
        left_shoulder.y + right_shoulder.y
    ) / 2


    hip_center_x = (
        left_hip.x + right_hip.x
    ) / 2

    hip_center_y = (
        left_hip.y + right_hip.y
    ) / 2


    # ==========================================
    # 1. eye_shoulder_y_ratio
    # ==========================================

    eye_shoulder_y_ratio = (
        eye_center_y /
        (shoulder_center_y + 1e-6)
    )


    # ==========================================
    # 2. shoulder_y_diff
    # ==========================================

    shoulder_y_diff = (
        left_shoulder.y -
        right_shoulder.y
    )


    # ==========================================
    # 3. wrist_distance_x
    # ==========================================

    wrist_distance_x = (
        left_wrist.x -
        right_wrist.x
    )


    # ==========================================
    # 4. wrist_shoulder_ratio
    # ==========================================

    shoulder_span = math.sqrt(
        (left_shoulder.x - right_shoulder.x) ** 2 +
        (left_shoulder.y - right_shoulder.y) ** 2
    )

    wrist_distance = math.sqrt(
        (left_wrist.x - right_wrist.x) ** 2 +
        (left_wrist.y - right_wrist.y) ** 2
    )

    wrist_shoulder_ratio = (
        wrist_distance /
        (shoulder_span + 1e-6)
    )


    # ==========================================
    # 5. nose_eye_center_offset_x
    # ==========================================

    nose_eye_center_offset_x = (
        nose.x -
        eye_center_x
    )


    # ==========================================
    # 6. shoulder_span
    # ==========================================

    shoulder_span = shoulder_span


    # ==========================================
    # 7. hip_shoulder_y_diff
    # ==========================================

    hip_shoulder_y_diff = (
        hip_center_y -
        shoulder_center_y
    )


    # ==========================================
    # 8. body_lean_x
    # ==========================================

    body_lean_x = (
        shoulder_center_x -
        hip_center_x
    )


    # ==========================================
    # 9. shoulder_center_x
    # ==========================================

    shoulder_center_x = shoulder_center_x


    # ==========================================
    # 10. hip_center_x
    # ==========================================

    hip_center_x = hip_center_x


    # ==========================================
    # 11. spine_angle
    # ==========================================

    dx = shoulder_center_x - hip_center_x
    dy = shoulder_center_y - hip_center_y

    spine_angle = math.degrees(
        math.atan2(dx, -dy)
    )


    # ==========================================
    # 12. eye_distance
    # ==========================================

    eye_distance = math.sqrt(
        (left_eye.x - right_eye.x) ** 2 +
        (left_eye.y - right_eye.y) ** 2
    )


    # ==========================================
    # 13. head_tilt_angle
    # ==========================================

    head_tilt_angle = math.degrees(
        math.atan2(
            left_eye.y - right_eye.y,
            left_eye.x - right_eye.x
        )
    )


    # ==========================================
    # 14. eye_distance_ratio
    # ==========================================

    eye_distance_ratio = (
        eye_distance /
        (shoulder_span + 1e-6)
    )


    # ==========================================
    # 15. shoulder_slope
    # ==========================================

    shoulder_slope = (
        (left_shoulder.y - right_shoulder.y) /
        (left_shoulder.x - right_shoulder.x + 1e-6)
    )


    # ==========================================
    # 16. head_direction
    # ==========================================

    head_direction = (
        nose.x -
        eye_center_x
    )


    # ==========================================
    # 17. arm_position
    # ==========================================

    left_arm_length = math.sqrt(
        (left_wrist.x - left_shoulder.x) ** 2 +
        (left_wrist.y - left_shoulder.y) ** 2
    )

    right_arm_length = math.sqrt(
        (right_wrist.x - right_shoulder.x) ** 2 +
        (right_wrist.y - right_shoulder.y) ** 2
    )

    arm_position = (
        left_arm_length +
        right_arm_length
    ) / 2


    # ==========================================
    # 18. posture
    # ==========================================

    posture = abs(spine_angle)


    # ==========================================
    # RETURN ALL 18 FEATURES
    # ==========================================

    return [
        eye_shoulder_y_ratio,       # 1
        shoulder_y_diff,            # 2
        wrist_distance_x,           # 3
        wrist_shoulder_ratio,       # 4
        nose_eye_center_offset_x,   # 5
        shoulder_span,              # 6
        hip_shoulder_y_diff,        # 7
        body_lean_x,                # 8
        shoulder_center_x,          # 9
        hip_center_x,               # 10
        spine_angle,                # 11
        eye_distance,               # 12
        head_tilt_angle,            # 13
        eye_distance_ratio,         # 14
        shoulder_slope,             # 15
        head_direction,             # 16
        arm_position,               # 17
        posture                     # 18
    ]

In [9]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

features = None

print("Look at the camera...")
print("Press Q after a few seconds")

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    results = pose.process(rgb_frame)

    if results.pose_landmarks:

        features = extract_features(
            results.pose_landmarks.landmark
        )

        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

        cv2.putText(
            frame,
            "18 Features Detected",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    cv2.imshow(
        "Feature Test",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


if features is not None:

    print("Number of features:", len(features))
    print("Features:")
    
    for i, value in enumerate(features, 1):
        print(f"{i}. {value}")

else:

    print("No landmarks detected")

Look at the camera...
Press Q after a few seconds
Number of features: 18
Features:
1. 0.5032236254169683
2. 0.014835774898529053
3. 0.9807152096182108
4. 1.4577788542711594
5. -0.00013358891010284424
6. 0.684051643791126
7. 1.1339807212352753
8. -0.0031884536147117615
9. 0.45893803983926773
10. 0.4621264934539795
11. -0.16110014081421486
12. 0.14893676156259206
13. 3.0492262758684165
14. 0.21772704617755934
15. 0.021693162706006748
16. -0.00013358891010284424
17. 0.972435135058489
18. 0.16110014081421486


In [10]:
if features is None:
    print("No features available.")
else:

    probabilities = model.predict_proba(
        [features]
    )[0]

    print("Classes:", model.classes_)
    print("Probabilities:", probabilities)

    if 1 in model.classes_:

        confidence_index = list(
            model.classes_
        ).index(1)

        confidence = (
            probabilities[confidence_index] * 100
        )

        print(
            f"Confidence: {confidence:.2f}%"
        )

    else:

        print(
            "Class 1 was not found in the model."
        )

Classes: [0 1 2]
Probabilities: [4.050785e-05 8.249964e-01 1.749631e-01]
Confidence: 82.50%


In [11]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")


# Store confidence predictions
confidence_scores = []

# Predict every 0.5 seconds
PREDICTION_INTERVAL = 0.5

last_prediction_time = 0

current_confidence = None


print("======================================")
print("       AI INTERVIEW STARTED")
print("======================================")
print("Press Q to finish the interview")


while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break


    # Flip camera
    frame = cv2.flip(frame, 1)


    # Convert to RGB
    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # MediaPipe
    results = pose.process(
        rgb_frame
    )


    # ==================================================
    # LANDMARKS DETECTED
    # ==================================================

    if results.pose_landmarks:

        # Draw landmarks

        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )


        current_time = time.time()


        # ==================================================
        # PREDICT EVERY 0.5 SECOND
        # ==================================================

        if (
            current_time - last_prediction_time
            >= PREDICTION_INTERVAL
        ):

            try:

                # Get 18 features

                features = extract_features(
                    results.pose_landmarks.landmark
                )


                # Make sure we have 18

                if len(features) != 18:

                    raise ValueError(
                        f"Expected 18 features, "
                        f"got {len(features)}"
                    )


                # XGBoost

                probabilities = (
                    model.predict_proba(
                        [features]
                    )[0]
                )


                # Get class 1 probability

                if 1 in model.classes_:

                    confidence_index = list(
                        model.classes_
                    ).index(1)

                    current_confidence = (
                        probabilities[
                            confidence_index
                        ] * 100
                    )

                else:

                    current_confidence = (
                        np.max(probabilities) * 100
                    )


                # Save result

                confidence_scores.append(
                    current_confidence
                )


                last_prediction_time = (
                    current_time
                )


            except Exception as e:

                print(
                    "Prediction error:",
                    e
                )


    # ==================================================
    # DISPLAY CONFIDENCE
    # ==================================================

    if current_confidence is not None:

        cv2.putText(
            frame,
            f"Confidence: {current_confidence:.1f}%",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

    else:

        cv2.putText(
            frame,
            "Confidence: Detecting...",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2
        )


    # ==================================================
    # NUMBER OF SAMPLES
    # ==================================================

    cv2.putText(
        frame,
        f"Samples: {len(confidence_scores)}",
        (30, 90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ==================================================
    # INSTRUCTION
    # ==================================================

    cv2.putText(
        frame,
        "Press Q to finish",
        (30, frame.shape[0] - 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )


    # ==================================================
    # SHOW CAMERA
    # ==================================================

    cv2.imshow(
        "AI Interview Confidence",
        frame
    )


    # ==================================================
    # QUIT
    # ==================================================

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


# ======================================================
# CLOSE CAMERA
# ======================================================

cap.release()
cv2.destroyAllWindows()


# ======================================================
# FINAL RESULTS
# ======================================================

print()
print("======================================")
print("       INTERVIEW FINISHED")
print("======================================")


if len(confidence_scores) > 0:

    average_confidence = np.mean(
        confidence_scores
    )

    minimum_confidence = np.min(
        confidence_scores
    )

    maximum_confidence = np.max(
        confidence_scores
    )


    print(
        f"Number of predictions: "
        f"{len(confidence_scores)}"
    )

    print(
        f"Average Confidence: "
        f"{average_confidence:.2f}%"
    )

    print(
        f"Minimum Confidence: "
        f"{minimum_confidence:.2f}%"
    )

    print(
        f"Maximum Confidence: "
        f"{maximum_confidence:.2f}%"
    )

    print("======================================")

else:

    print("No confidence predictions collected.")
    print("Make sure your body is visible to the camera.")

       AI INTERVIEW STARTED
Press Q to finish the interview

       INTERVIEW FINISHED
Number of predictions: 13
Average Confidence: 87.81%
Minimum Confidence: 77.73%
Maximum Confidence: 96.48%
